# Fobench Basics
Welcome to the tutorial of the basic functionalities of Fobench. This notebook will guide you through the loading of data, preprocessing steps and visuals. Of course we can not go through every method in detail, so feel free to dive into the documentation if you want to know more. Here and there we will hint to more functionality to discover but for now we will stick to the basics.

A note about plotting: 
Most methods have two plotting modes: matplotlib (plot_mode='mpl') and PyQt (plot_mode='pyqt'), the standard in Fobench is using PyQt as it performs better when it comes to larger matrix plots. However it is not very stable when working in a jupyter notebook, it should however work from other IDEs.


In [1]:
import os
os.environ['PYQTGRAPH_QT_LIB'] = 'PyQt5'
from fobench.fiber import Fiber
%gui qt

The workhorse of Fobench is the `Fiber` class, it contains the actual fiber optic record with all its metadata and provides a lot of functionality to manipulate data in space and time. Let us go right ahead and load a file. We simply give the path to the file and we let Fobench know the manufacturer of the interrogator we used. In our case we will use data recorded on an Aragon Photonics HDAS system.

In [ ]:
das = Fiber('./example_data/aragon_h5/2024_07_11_05h15m16s_HDAS_2DRawData_Strain.h5', 'aragon')

Read File ✓: 100%|██████████| 1/1 [00:00<00:00,  1.13it/s]               


The following other manufacturers are supported in Fobench:

    - 'tmds' from 'silixa'
    - 'hdf5' from 'silixa'
    - 'hdf5' from 'febus'
    - 'hfd5' from 'terra15'
    - 'hdf5' from 'asn'
    - 'hdf5' from 'quantx'
    - 'h5'   from 'sintela'

First, we will have a look at the most important aquisition parameters of our file:

In [3]:
print(das)
# das.metadata() # for the full metadata

Instance of Fiber class
recording parameters:
-----------------------------------------------------------------
units                     = strain
start_time                = 2024-07-11T05:15:15.791536Z
end_time                  = 2024-07-11T05:16:15.791536Z
num_points                = 15000
total_channels            = 501
spatial_interval          = 1.0
sampling_frequency        = 250.0
gauge_length              = 6.0


We can see that the file contains only a single minute of strain data, lets concatenate a second file and convert into strain-rate after. 

In [ ]:
das += Fiber('./example_data/aragon_h5/2024_07_11_05h16m16s_HDAS_2DRawData_Strain.h5', 'aragon') # the += syntax just calls Fiber.concatenate
das.differentiate() # Fiber.integrate for integration of the data

Read File ✓: 100%|██████████| 1/1 [00:01<00:00,  1.02s/it]               


Instance of Fiber class
recording parameters:
-----------------------------------------------------------------
units                     = strain-rate
start_time                = 2024-07-11T05:15:15.791536Z
end_time                  = 2024-07-11T05:17:15.791002Z
num_points                = 30000
total_channels            = 501
spatial_interval          = 1.0
sampling_frequency        = 250.0
gauge_length              = 6.0

Our original sampling frequency is a bit too high, we will decimate the data to 50 Hz.

If we then check the aquisition parameters again, we can see that we now have two minutes of strain-rate data at 50 Hz

In [10]:
new_freq = 50
das.decimate(new_freq)
print(das)

Instance of Fiber class
recording parameters:
-----------------------------------------------------------------
units                     = strain-rate
start_time                = 2024-07-11T05:15:15.791536Z
end_time                  = 2024-07-11T05:17:15.791002Z
num_points                = 5978
total_channels            = 501
spatial_interval          = 1.0
sampling_frequency        = 50.0
gauge_length              = 6.0


The `Fiber` class stores the actual data in the `Fiber.data` attribute. This way we can easily access and extract it:

In [24]:
print(type(das.data), das.data.shape, das.data)
#das.get_data() # returns full data or data of a specific channel
#das.times # returns the time stamps for each sample in specified format

<class 'numpy.ndarray'> (30000, 501) [[-2.41120833e-06 -2.26114583e-06 -1.78952083e-06 ...  2.68479167e-07
   1.14741667e-06 -9.59583333e-08]
 [-1.40607031e-06 -1.34175781e-06 -3.02039062e-07 ...  1.37429688e-07
   8.66304688e-07  7.48398438e-07]
 [ 9.45036458e-07  7.84255208e-07  8.59286458e-07 ... -2.44744792e-07
   1.25052083e-08  9.34317708e-07]
 ...
 [ 9.20026042e-08  4.88596354e-07  7.99440104e-07 ...  8.20877604e-07
   1.56315104e-07 -2.59036458e-08]
 [ 2.04166667e-09  2.91447917e-07  4.41510417e-07 ...  4.73666667e-07
   2.16416667e-07  1.52104167e-07]
 [ 2.65671875e-07 -5.58906250e-08 -6.99015625e-07 ...  1.15609375e-07
   3.72859375e-07  1.37046875e-07]]


Most operations on the data are done inplace. Just in case we mess something up later on, at any point we can get a copy of the records current state with `Fiber.copy()`

In [25]:
backup_das = das.copy()

Before we have a look at the data, we should apply some basic preprocessing. Let us start with a simple bandpass filter between 0.1 and 20 Hz. We can check if the filter was applied successfully by looking at the list stored in `Fiber.processing`:

In [14]:
das.filter(f_type='bandpass', freq=(0.1, 20)) # other filter options are highpass, lowpass and bandstop. Fobench also provides more specialized filters such as median, Cheby and FIR-filters
das.processing

[{'instance creation': 'Tue Feb  3 16:55:05 2026'},
 {'differentiate': {'method': 'gradient', 'dim': 't'}},
 {'decimate': {'new_freq': 50, 'f_type': 'fir-remez'}},
 {'preprocess': {'alpha': 0.05,
   'order': 1,
   'sym': True,
   'axis': 0,
   'steps': (True, True, True)}},
 {'filter': {'f_type': 'bandpass',
   'freq': (0.1, 20),
   'pre_process': True,
   'alpha': 0.05,
   'order': 1,
   'sym': True,
   'options': {}}}]

The list keeps track of all preprocessing steps included in Fobench so that at any point we can go back and check which steps we took in what order. This is done by storing the methods name and all parameters it is called with.

We can see that before bandpass filtering the data was detrended, demeaned and tapered. That is because the filter function by default conveniently performs these steps before the actual filtering. Of course all these methods can be used individually and `filter` can be used with the `pre_process` parameter set to `False`.

Note also that most methods have additional parameters such as the removal of higher order polynomial trends by passing the `order` to `Fiber.detrend()`. A lot of methods can be applied in both space and time, this is determined by the `dim` parameter.

Now back to our data! Let us plot the record in both time and frequency domain:

In [6]:
das.plot()
das.fx_plot()

It seems that we have recorded a small event! However it looks like towards the end of the cable we have no useful data anymore. We will trim the record and focus on the channels 20 to 280

In [27]:
das.restrict_channels(20, 280)
das.total_channels # number of channels in the record
# print(das.channels_num) # all channel numbers

261

Now that we have indentified the channels that we are interested in, the next time when we are loading data, we can retrieve only the part of interest. This can decrease the loading time significantly. We do that by simply passing the channel range to the Fiber class: 
```
file = './example_data/2024_07_11_05h15m16s_HDAS_2DRawData_Strain.h5'
manufacturer = 'aragon'
channels = [20, 280]
das = Fiber(file, manufacturer, range_ch=channels)
```
We can also trim the record around the time of the event and normalize to the absolute max of the record before plotting:

In [28]:
t0 = das.start_time + 43
tf = t0 + 16
das.trim(t0, tf)
das.normalize() #default is abolute max normalization, other options are trace max, running absolute mean and 1bit
das.plot()

We can look at a few channels in more detail, lets go for 100 - 120:

In [29]:
section = (100, 120)
das.record_section(section)

We can plot a single channels waveform and the corresponding spectrogram and amplitude spectrum:

In [30]:
ch = 105
das.channel_plot(ch)
das.channel_spectrogram(ch)
das.spectrum(channel=ch, mode='psd')

Render time: 691.3078 s


To explore the data a bit deeper we can call the data explorer:

In [34]:
das.explore()

-----------------------------------------------------------------
Starting Fobench Data Explorer
-----------------------------------------------------------------


We can also explore the spatial coherence of the data and the autocorrelation of the cable segment:

In [33]:
das.spatial_coherence(max_lag=1)

In [47]:
das.acf_profile(max_lag=5)